# Домашнее задание - линейная регрессия

### Работа с признаками (8 баллов)

Скачайте датасет из материалов к уроку или по ссылке https://raw.githubusercontent.com/jupiterzhuo/travel-insurance/master/travel%20insurance.csv 


Описание признаков:

* Agency — название страхового агентства
* Agency Type — тип страхового агентства
* Distribution Channel — канал продвижения страхового агентства
* Product Name — название страхового продукта
* Duration — длительность поездки (количество дней)
* Destination — направление поездки
* Net Sales — сумма продаж 
* Commission (in value) — комиссия страхового агентства
* Gender — пол застрахованного
* Age — возраст застрахованного

Ответ:
* Claim — потребовалась ли страховая выплата: «да» — 1, «нет» — 0

Обработайте пропущенные значения и примените написанные функции onehot_encode() и minmax_scale().

**Подсказка**: маску для категориальных признаков можно сделать фильтром cat_features_mask = (df.dtypes == "object").values

In [ ]:
# импортирум все необходимое
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
#"примените написанные функции" не совсем понял, но на всякий случай буду использовать функции из прошлых заданий
def onehot_encoding(x: np.ndarray):
    L = len(x)
    x1 = np.sort(x)
    w = {}
    N = 0
    for j in x1:
        if j not in w:
            w[j] = N
            N += 1

    f = np.eye(N, N, dtype = np.float32)
    Q = np.empty([L, N], np.float32)
    for j, i in enumerate(x):
        Q[j] = f[w[i]]

    return Q


def minmax_scale(X: np.ndarray):
    if len(X) <= 1:
        return np.zeros_like(X, np.float32)
    Mi = X.min(axis=0)
    Ma = X.max(axis=0)
    return (X - Mi) / (Ma - Mi)




def get_claim_info(value_name, print_info = True): # показывает долю страховых выплат по каждому значению категориального параметра
    global df
    VU = df[value_name].unique()
    HH = {}
    for v in VU:
        s = len(df[(df[value_name] == v) & (df['Claim'] == 'Yes')])/len(df[df[value_name]==v])
        if print_info:print(f"{v} : {100*s}%")
        HH[v] = s

    return HH

def show_all(print_info = True):
    q = list(df.dtypes[df.dtypes == "object"].keys())
    q.remove('Claim')
    for g in q:
        print(f"============ {g} ==================")
        HH = get_claim_info(g)
        plt.pie(HH.values(), labels = HH.keys())
        plt.show()


#### Обрабатываю NaN

In [ ]:
df = pd.read_csv("travel insurance.csv")

In [ ]:
# первым делом посмотрим сколько промущенных значений встречается в данных
Nans = df[df.isnull().any(axis=1)] # берем все строки в которых есть хотя бы 1 Nan
RowCount = df.shape[0]
NaNRowCount = Nans.shape[0]

NaNRowCount/RowCount

In [ ]:
# теперь посмотрим сколько Nan - ов в каждой коллонке
df.isnull().sum(axis=0)

In [ ]:
# Как можно заметить все Наны содержатся в колонке Gender
# 71% данных в этой колонке неизвестны, будет разумно избавиться от неё

df = df.drop(columns='Gender', axis=1)
df

#### Подробный анализ и подготовка датасета

Подробный анализ и подготовка датасета часто помогают улучшить качество модели. Ниже представлено несколько идей преобразований. Вы можете применить одно или несколько из этих преобразований (а можете не применять), чтобы помочь будущей модели. 

1. Посмотрите на количественные признаки. Возможно, в некоторых признаках есть выбросы - значения, которые сильно выбиваются. Такие значения полезно удалять. Советуем присмотреться к колонке Duration)

2. Можно заметить, что one hot encoding сильно раздувает количество столбцов. Радикальное решение - можно попробовать выбросить все категориальные признаки из датасета.

3. Если все-таки оставляете категориальные признаки, то подумайте, как уменьшить количество столбцов после one hot encoding. Признаки с большим количеством значений (Duration - 149! разных стран) можно удалить или попробовать сгруппировать некоторые значения.

4. Downsampling. Датасет достаточно большой, разница в классах огромная. Можно уменьшить число наблюдений с частым ответом.

##### DOWNSAMPLING

Прежде стоит посмотреть соотношения Claim для True/False

In [ ]:
C = len(df['Claim'])
print(f'True: {len(df[df["Claim"] == "Yes"])}')
print(f"False: {len(df[df['Claim'] == 'No'])}")
print(f'True/C: {100*len(df[df["Claim"] == "Yes"])/C}%')
print(f"False/C: {100*len(df[df['Claim'] == 'No'])/C}%")
print(f"True/False: {100*len(df[df['Claim'] == 'Yes'])/len(df[df['Claim'] == 'No'])}%")

В данных и в правду большенство значений для Claim = 0, по этому стоит убрать (62399 - 927 = 61472) строк со значением Claim = 0

Строки выберем случайно, что бы по минимуму повлиять на зависимости

In [ ]:
print('до')

show_all()

n = 61472

df = df.sample(frac=1, random_state=123) # на вcякий случай перемешаем порядок строк
#df[df['Claim'] == 'No'] = df[df['Claim'] == 'No'].reset_index(drop = True)


df = df.drop(df[df['Claim'] == 'No'].head(n).index)

print("----------------- после ----------------------------")
show_all()


print("-----------------------------------------------------------------------")

C = len(df['Claim'])
print(f'True: {len(df[df["Claim"] == "Yes"])}')
print(f"False: {len(df[df['Claim'] == 'No'])}")
print(f'True/C: {100*len(df[df["Claim"] == "Yes"])/C}%')
print(f"False/C: {100*len(df[df['Claim'] == 'No'])/C}%")
print(f"True/False: {100*len(df[df['Claim'] == 'Yes'])/len(df[df['Claim'] == 'No'])}%")

Как и стоило ожидать, после установления балланса между yes и no, данные стали более равномерны по отношениию к claim, хотя и не полностью, значени категориальных признаков все еще можно упорядочить

##### Применяю Функции 

Перед подготовкой данных нужно понять, а сколько различных значений у каждого признака есть

###### Фильтруем признаки

In [ ]:
df.select_dtypes("object").nunique()

Будет размуно посмотреть как связаны категориальные признаки и Claim

In [ ]:
def get_claim_info(value_name, print_info = True): # показывает долю страховых выплат по каждому значению категориального параметра
    global df
    VU = df[value_name].unique()
    HH = {}
    for v in VU:
        s = len(df[(df[value_name] == v) & (df['Claim'] == 'Yes')])/len(df[df[value_name]==v])
        if print_info:print(f"{v} : {100*s}%")
        HH[v] = s

    return HH

def show_all(print_info = True):
    q = list(df.dtypes[df.dtypes == "object"].keys())
    q.remove('Claim')
    for g in q:
        print(f"============ {g} ==================")
        HH = get_claim_info(g)
        plt.pie(HH.values(), labels = HH.keys())
        plt.show()

In [ ]:
show_all()


1) Как можно заметить, некоторые категориальные признаки, вроде стран, можно заменить количественными (вероятность того что при путишествии в эту страну придется выплатить страховку)
2) агенства тоже можно заменить на количественый признак
3) страховой план тоже можно заменить количественным признаком
4) Тип агенства можно не менять - нет нужды


In [ ]:
level_map = get_claim_info('Destination', False)
df['Destination'] = df['Destination'].map(level_map)

level_map = get_claim_info('Agency', False)
df['Agency'] = df['Agency'].map(level_map)

level_map = get_claim_info('Product Name', False)
df['Product Name'] = df['Product Name'].map(level_map)

Таким образом получилось учитывать категориальные признаки, но только в том объеме в котором они нам нужны

###### Обработка числовых признаков

Посмотрим на числовые признаки

In [ ]:
def show_all_count(key = None):
    q = list(df.dtypes[df.dtypes != "object"].keys())
    c = not (key is None)
    fig, ax = plt.subplots(1, len(q), figsize=(45, 6))
    for n, i in enumerate(q):
        ax[n].plot(key(df[i]) if c else df[i])
        ax[n].set_title(i + (f" ({key.__name__})" if c else ''))
        ax[n].grid(True)
    plt.show()

show_all_count()
show_all_count(sorted) # так же выведем в отсортированном порядке, что бы наглядно увидеть долю выбросов

In [ ]:
### убираем очевидные выбросы
df = df[df['Duration'] < 200]
df = df[(df['Net Sales'] < 110) & (df['Net Sales'] > 0)]
df = df[df['Commision (in value)'] < 80]
df = df[(df['Age'] < 70) & (df['Age'] >= 20)]

show_all_count()

##### Конвертирую в numpy массивы

In [ ]:
# теперь можно конвертировать обработаные данные в numpy массив и применить к нему 'написанные функции'

cat_features_mask = (df.dtypes == "object").loc[df.columns != 'Claim'].values
DX = df.loc[:, df.columns != 'Claim'].to_numpy()
X = []
y = np.where(df['Claim'].to_numpy() == 'Yes', 1.0, 0.0)

for n, i in enumerate(cat_features_mask):
    if i:
        X.append(onehot_encoding(DX[:, n]).transpose())
    else:
        X.append(minmax_scale(DX[:, n]))

X = np.vstack(X).transpose()
X = np.matrix(X, dtype=np.float64)
print(X)
print(y)

### Применение линейной регрессии (10 баллов)

Это задача классификации, но её можно решить с помощью линейной регрессии, если округлять предсказанный ответ до целого и выбирать ближайший по значению ответ из множества {0, 1}.

Вынесите признак 'Claim' в вектор ответов и разделите датасет на обучающую и тестовую выборку в соотношении 80 к 20. Зафиксируйте random_state.

**Подсказка:** быстро перевести Yes/No в 1/0 можно так - np.where(df['Claim'] == 'Yes', 1,0)

In [ ]:
# разделение на test/train
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=228)

Найдите аналитическое решение для обучающей выборки: обычное и регуляризацией l2. 

In [3]:
import numpy as np

def Q(x): # функция риска
    return x @ x.T

# посчитайте аналитическое решение

class LinearRegression:
    def __init__(self, biases:bool = True) -> None:
        self.biases = biases
        self.w = None
    
    def fit(self, X_train, y_train):
        Xt = X_train
        if self.biases:
            Xt = np.hstack((np.ones((Xt.shape[0], 1)), X_train))
        
        self.w = np.linalg.inv(Xt.transpose() @ Xt) @ Xt.transpose() @ y_train
        self.w = self.w.T

    def calculate(self, x):
        if not self.w is None:
            tx = np.hstack([np.ones((x.shape[0], 1)), x]) if self.biases else x
            return np.array((tx @ self.w).T)
        

X_train = np.matrix(
    [
    [1, 9, 6, 3],
    [3, 1, 9, 6],
    [6, 3, 1, 9],
    [9, 6, 3, 1],
    [1, 0, 1, 0]]
)
y_train = np.array([2, 1, 3, -1, 1])

Lr = LinearRegression()
Lr.fit(X_train, y_train)
np.round(Lr.w, 2)

array([[ 1.53],
       [-0.32],
       [ 0.1 ],
       [-0.21],
       [ 0.37]])

In [ ]:
Lr.w # значение весов

In [ ]:
o # значение выхода

In [ ]:
Q(o - y_test)/len(y_test) # MSE

In [ ]:
Q(np.round(o) - y_test)/len(y_test) # MSE с округлеными ответами

In [5]:
import numpy as np
# посчитать аналитическое решение с регуляризацией

class LinearRegressionWithRegularization:
    def __init__(self, biases:bool = True) -> None:
        self.biases = biases
        self.w = None
    
    def fit(self, X_train, y_train, _lambda = 1):
        Xt = X_train
        if self.biases:
            Xt = np.hstack((np.ones((Xt.shape[0], 1)), X_train))
        
        I = np.eye(Xt.shape[1] + 1) if self.biases else np.eye(Xt.shape[1])
        self.w = np.linalg.inv(Xt.transpose() @ Xt + _lambda**2 * I) @ Xt.transpose() @ y_train
        self.w = self.w.transpose()

    def calculate(self, x):
        if not self.w is None:
            tx = np.hstack([np.ones((x.shape[0], 1)), x]) if self.biases else x
            return np.array((tx @ self.w).T)

X_train = np.matrix(
    [
    [1, 9, 6, 3],
    [3, 1, 9, 6],
    [6, 3, 1, 9],
    [9, 6, 3, 1],
    [1, 0, 1, 0]]
)
y_train = np.array([2, 1, 3, -1, 1])


Lr = LinearRegressionWithRegularization(False)
Lr.fit(X_train, y_train)
np.round(Lr.w, 2)

array([[-0.2 ],
       [ 0.15],
       [-0.09],
       [ 0.41]])

In [ ]:
Lr.w # значение весов

In [ ]:
o # значение выхода

In [ ]:
Q(o - y_test)/len(y_test) # MSE

In [ ]:
Q(np.round(o) - y_test)/len(y_test) # MSE с округлеными ответами

Постройте модель LinearRegression, примените к тестовой выборке и посчитайте MSE (можно использовать библиотеку sklearn)

In [ ]:
# обучите модель линейной регрессии LinearRegression на обучающей выборке, примените к тестовой
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lr = LinearRegression()
lr.fit(np.array(X_train), np.array(y_train))
o = lr.predict(np.array(X_test))
o


In [ ]:
# посчитайте MSE, предварительно округлив предсказанные ответы до целого
mean_squared_error(y_test, np.round(o))

### Вывод (1 балла)

Напишите краткий вывод по заданию (достаточно пары предложений). Расскажите, какие способы предобработки данных вы выбрали и почему. Насколько хороша ваша модель?

Я сопоставил категориальным признакам, с большим количествам разных значений - числа, которые косвенно отражают вероятность claim (Так можно сделать т.к если эта выборка достаточно объективна, то страны/агенства/тип страховки можно упорядочить по рейтингу и в этом контексте " значения категории уже не будут линейно независимы" ;) ).
Моя модель хорошая, т.к даже без регулизации MSE достаточно маленькая - думаю это связано с большим количеством независимых друг от друга параметров (возраст с длительность и т.д), а так же с тем, что я сократил количество категориальных признаков до минимума, т.к 1 категориальный признак превращается в множество зависимых друг от друга численых параметров (если объект принадлежит к категори A = (1, 0), то он точно не пренадлежит к категории B = (0, 1))

Минус модели - это в первую очередь факт сильной предобработки данных, т.е если взять произвольные значения для признаков, то их вначале придется привести к виду, который модель поймет (хотя это повышает интерпритируемость модели)

Ляшенков Илья